In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import copy
import time

# --- Configuration ---
DATA_DIR = '/home/sagemaker-user/AI_Matrix_Project/Dataset/banana-ripeness-dataset-original'
BATCH_SIZE = 16
NUM_EPOCHS = 10
NUM_CLASSES = 6  # updated to 6: freshrip, freshunrip, overripe, ripe, rotten, unrip
INPUT_SIZE = 224
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# --- 1. Data Augmentation & Loading ---
# Mapping your specific augmentation requirements to PyTorch transforms
train_transforms = transforms.Compose([
    # Crop: 0% Min Zoom, 20% Max Zoom (Scaling between 80% and 100% of the image)
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.8, 1.0)),
    
    # Flip: Horizontal & Vertical
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    
    # 90° Rotate (CW, CCW, Upside down) - Randomly picking 90, -90, or 180 degrees
    transforms.RandomChoice([
        transforms.RandomRotation((90, 90)),
        transforms.RandomRotation((-90, -90)),
        transforms.RandomRotation((180, 180)),
        transforms.RandomRotation((0, 0)) 
    ]),
    
    # Rotation: Between -15° and +15°
    transforms.RandomRotation(degrees=[-15, 15]),
    
    # Hue (-10° to 10°), Saturation (-10% to 10%), Brightness (-10% to 10%), Exposure (-10% to 10%)
    # Note: Exposure is represented by 'contrast' in standard torchvision transforms
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    
    # Blur: Up to 1px (using GaussianBlur with a small sigma and 3x3 kernel)
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0)),
    
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

# Validation & Test transforms (No augmentation, just resize and normalize)
val_test_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), train_transforms),
    'valid': datasets.ImageFolder(os.path.join(DATA_DIR, 'valid'), val_test_transforms),
    'test': datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), val_test_transforms)
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=True if x == 'train' else False, num_workers=4)
    for x in ['train', 'valid', 'test']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'valid', 'test']}
class_names = image_datasets['train'].classes
print(f"Classes found ({len(class_names)}): {class_names}")

# --- 2. Model Training Function ---
def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model if it has the best validation accuracy
            if phase == 'valid' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

# --- 3. Model Setup & Execution ---
# Using the default (and most recent) weights for transfer learning
models_to_train = {
    "MobileNetV3": models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT),
    "EfficientNetB0": models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT),
    "ResNet50": models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
}

trained_models = {}

for model_name, model in models_to_train.items():
    print(f"\n{'='*40}\nTraining {model_name}\n{'='*40}")
    
    # Replace the final classification layer to match our number of classes (6)
    if model_name == "ResNet50":
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
    elif model_name == "MobileNetV3":
        num_ftrs = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(num_ftrs, NUM_CLASSES)
    elif model_name == "EfficientNetB0":
        num_ftrs = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_ftrs, NUM_CLASSES)

    model = model.to(DEVICE)
    
    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Train the model
    trained_model = train_model(model, criterion, optimizer, num_epochs=NUM_EPOCHS)
    trained_models[model_name] = trained_model
    
    # Save the model artifact
    torch.save(trained_model.state_dict(), f'{model_name}_banana_ripeness.pth')
    print(f"Saved {model_name} weights to {model_name}_banana_ripeness.pth")

# --- 4. Evaluate on Test Set ---
print("\nFinal Evaluation on Test Set:")
for model_name, model in trained_models.items():
    model.eval()
    running_corrects = 0
    with torch.no_grad():
        for inputs, labels in dataloaders['test']:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            
    test_acc = running_corrects.double() / dataset_sizes['test']
    print(f"{model_name} Test Accuracy: {test_acc:.4f}")

Using device: cuda:0
Classes found (6): ['freshripe', 'freshunripe', 'overripe', 'ripe', 'rotten', 'unripe']

Training MobileNetV3
Epoch 1/10
----------
Train Loss: 0.4048 Acc: 0.8631
Valid Loss: 0.2326 Acc: 0.9288

Epoch 2/10
----------
Train Loss: 0.2236 Acc: 0.9255
Valid Loss: 0.1754 Acc: 0.9421

Epoch 3/10
----------
Train Loss: 0.1690 Acc: 0.9473
Valid Loss: 0.1673 Acc: 0.9528

Epoch 4/10
----------
Train Loss: 0.1646 Acc: 0.9501
Valid Loss: 0.3654 Acc: 0.8762

Epoch 5/10
----------
Train Loss: 0.1190 Acc: 0.9621
Valid Loss: 0.2717 Acc: 0.9484

Epoch 6/10
----------
Train Loss: 0.1306 Acc: 0.9573
Valid Loss: 0.1453 Acc: 0.9546

Epoch 7/10
----------
Train Loss: 0.1377 Acc: 0.9560
Valid Loss: 0.1219 Acc: 0.9671

Epoch 8/10
----------
Train Loss: 0.1408 Acc: 0.9534
Valid Loss: 0.1639 Acc: 0.9590

Epoch 9/10
----------
Train Loss: 0.1120 Acc: 0.9646
Valid Loss: 0.3184 Acc: 0.9029

Epoch 10/10
----------
Train Loss: 0.1477 Acc: 0.9532
Valid Loss: 0.1214 Acc: 0.9679

Training complete 

In [2]:
import os
import torch

# 1. Create a directory to keep your models organized
save_dir = 'saved_models'
os.makedirs(save_dir, exist_ok=True)

print(f"Saving models to the '{save_dir}' directory...\n")

# 2. Iterate through the trained models dictionary and save each one
for model_name, model in trained_models.items():
    # Create the file path (e.g., saved_models/MobileNetV3_banana_ripeness.pth)
    save_path = os.path.join(save_dir, f'{model_name}_banana_ripeness.pth')
    
    # Save the model's weights (state_dict is the recommended PyTorch method)
    torch.save(model.state_dict(), save_path)
    
    print(f"✅ Successfully saved {model_name} -> {save_path}")

print("\nAll models are saved and ready for deployment or inference!")

Saving models to the 'saved_models' directory...

✅ Successfully saved MobileNetV3 -> saved_models/MobileNetV3_banana_ripeness.pth
✅ Successfully saved EfficientNetB0 -> saved_models/EfficientNetB0_banana_ripeness.pth
✅ Successfully saved ResNet50 -> saved_models/ResNet50_banana_ripeness.pth

All models are saved and ready for deployment or inference!


In [3]:
import os
print("Current Working Directory:", os.getcwd())
print("Files/Folders in this directory:", os.listdir('.'))

Current Working Directory: /home/sagemaker-user/AI_Matrix_Project
Files/Folders in this directory: ['README.md', 'Untitled.ipynb', '.git', 'dataset_storage.ipynb', '.ipynb_checkpoints', 'AI-Studio-ClearML', 'Dataset', 'Untitled1.ipynb']
